In [13]:
# I used pythons 3.12.13
import time

import torch
from torch.utils.data import DataLoader
from torchvision import datasets as tvd

from common import REPO_ROOT, get_device
import open_clip

SEED = 1
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print("seed:", SEED)

seed: 1


In [14]:
CKPT_PATH = REPO_ROOT / "tulip-so400m-14-384.ckpt"
MODEL_NAME = "TULIP-so400m-14-384"

DATASET = "oxford_pet"
# DATASET = "food101"
# DATASET = "dtd"
# DATASET = "aircraft"
DATA_ROOT = REPO_ROOT / "final-project" / "data"
OUT_ROOT = REPO_ROOT / "final-project" / "features" / DATASET
OUT_ROOT.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
NUM_WORKERS = 4

DEVICE = get_device()

AVAILABLE_DATASETS = {
        "oxford_pet": { "dataset": tvd.OxfordIIITPet, "train":"trainval", "test":"test"},
        "food101": { "dataset": tvd.Food101, "train":"train", "test":"test"},
        "dtd": { "dataset": tvd.DTD, "train":"train", "test":"test"},
        "aircraft": { "dataset": tvd.FGVCAircraft, "train":"train", "test":"test"},
    }

device: mps


In [15]:
model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=str(CKPT_PATH)
)
model.eval().to(DEVICE)
for param in model.parameters():
    param.requires_grad_(False)

img_size = model.visual.image_size
if isinstance(img_size, tuple):
    image_height, image_width = int(img_size[0]), int(img_size[1])
else:
    image_height = image_width = int(img_size)

with torch.no_grad():
    dummy = torch.zeros(1, 3, image_height, image_width, device=DEVICE)
    embed_dim = model.encode_image(dummy).shape[-1]
print("image_size:", (image_height, image_width), "embedding dim:", embed_dim)

image_size: (384, 384) embedding dim: 1152


In [ ]:
@torch.no_grad()
def extract(split):
    dataset_info = AVAILABLE_DATASETS[DATASET]
    dataset = dataset_info['dataset'](
        root=DATA_ROOT, split=dataset_info[split], download=True, transform=preprocess
    )

    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
    )
    num_samples = len(dataset)
    features = torch.empty(num_samples, embed_dim, dtype=torch.float16)
    labels = torch.empty(num_samples, dtype=torch.int64)

    samples_written = 0
    start_time = time.time()
    for batch_index, (images, batch_labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        batch_embeddings = model.encode_image(images).float()
        current_batch_size = batch_embeddings.shape[0]
        slice_end = samples_written + current_batch_size
        features[samples_written:slice_end] = batch_embeddings.cpu().to(torch.float16)
        labels[samples_written:slice_end] = batch_labels
        samples_written = slice_end
        if batch_index % 20 == 0:
            elapsed = time.time() - start_time
            print(f"  [{split}] {samples_written}/{num_samples}  ({elapsed:.1f}s)")

    total_seconds = time.time() - start_time
    print(f"  [{split}] done in {total_seconds:.1f}s — features {tuple(features.shape)}")
    return features, labels

In [6]:
for split in ("train", "test"):
    features, labels = extract(split)
    torch.save(features, OUT_ROOT / f"{split}_features.pt")
    torch.save(labels, OUT_ROOT / f"{split}_labels.pt")
    print(f"saved to {OUT_ROOT}/{split}_*.pt")

{'dataset': <class 'torchvision.datasets.oxford_iiit_pet.OxfordIIITPet'>, 'train': 'trainval', 'test': 'test'}
  [train] 32/3680  (19.6s)
  [train] 672/3680  (255.3s)
  [train] 1312/3680  (514.3s)
  [train] 1952/3680  (781.9s)
  [train] 2592/3680  (1055.0s)
  [train] 3232/3680  (1331.9s)
  [train] done in 1545.3s — features (3680, 1152)
saved to /Users/joshlichty/Repo/school/CS_547_open_clip_howeda_beanc_lichtyj/final-project/features/oxford_pet/train_*.pt
{'dataset': <class 'torchvision.datasets.oxford_iiit_pet.OxfordIIITPet'>, 'train': 'trainval', 'test': 'test'}
  [test] 32/3669  (21.6s)
  [test] 672/3669  (341.3s)
  [test] 1312/3669  (591.9s)
  [test] 1952/3669  (859.1s)
  [test] 2592/3669  (1172.5s)
  [test] 3232/3669  (1432.1s)
  [test] done in 1630.5s — features (3669, 1152)
saved to /Users/joshlichty/Repo/school/CS_547_open_clip_howeda_beanc_lichtyj/final-project/features/oxford_pet/test_*.pt


## Linear Probe

In [8]:
import json

from torch import nn
from torch.utils.data import TensorDataset

PROBE_EPOCHS = 20
PROBE_BATCH_SIZE = 512
PROBE_LR = 3e-3
PROBE_WEIGHT_DECAY = 1e-4

train_features = torch.load(OUT_ROOT / "train_features.pt").float()
train_labels = torch.load(OUT_ROOT / "train_labels.pt").long()
test_features = torch.load(OUT_ROOT / "test_features.pt").float()
test_labels = torch.load(OUT_ROOT / "test_labels.pt").long()

num_classes = int(train_labels.max().item()) + 1
print("loaded train:", tuple(train_features.shape), train_features.dtype)
print("loaded test: ", tuple(test_features.shape), test_features.dtype)
print("num_classes:", num_classes)

loaded train: (3680, 1152) torch.float32
loaded test:  (3669, 1152) torch.float32
num_classes: 37


In [10]:
train_mean = train_features.mean(dim=0)
train_std = train_features.std(dim=0).clamp_min(1e-6)

train_features_norm = (train_features - train_mean) / train_std
test_features_norm = (test_features - train_mean) / train_std

train_ds = TensorDataset(train_features_norm, train_labels)
test_ds = TensorDataset(test_features_norm, test_labels)

train_loader = DataLoader(train_ds, batch_size=PROBE_BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=PROBE_BATCH_SIZE, shuffle=False)

print("train batches:", len(train_loader))
print("test batches:", len(test_loader))

train batches: 8
test batches: 8


In [11]:
probe = nn.Linear(train_features.shape[1], num_classes).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    probe.parameters(),
    lr=PROBE_LR,
    weight_decay=PROBE_WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PROBE_EPOCHS)


def accuracy_topk(logits, labels, k=1):
    topk = logits.topk(k, dim=1).indices
    matches = topk.eq(labels.unsqueeze(1)).any(dim=1)
    return matches.float().mean().item()


@torch.no_grad()
def evaluate(loader):
    probe.eval()
    total_loss = 0.0
    total_count = 0
    total_top1 = 0.0
    total_top5 = 0.0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        logits = probe(x_batch)
        loss = criterion(logits, y_batch)

        batch_size = y_batch.shape[0]
        total_count += batch_size
        total_loss += loss.item() * batch_size
        total_top1 += accuracy_topk(logits, y_batch, k=1) * batch_size
        total_top5 += accuracy_topk(logits, y_batch, k=min(5, num_classes)) * batch_size

    return {
        "loss": total_loss / total_count,
        "top1": total_top1 / total_count,
        "top5": total_top5 / total_count,
    }


for epoch in range(1, PROBE_EPOCHS + 1):
    probe.train()
    running_loss = 0.0
    running_count = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE, non_blocking=True)
        y_batch = y_batch.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = probe(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.shape[0]
        running_count += batch_size
        running_loss += loss.item() * batch_size

    scheduler.step()

    train_loss = running_loss / running_count
    test_metrics = evaluate(test_loader)

    print(
        f"epoch {epoch:02d}/{PROBE_EPOCHS} "
        f"train_loss={train_loss:.4f} "
        f"test_loss={test_metrics['loss']:.4f} "
        f"test_top1={test_metrics['top1'] * 100:.2f}% "
        f"test_top5={test_metrics['top5'] * 100:.2f}%"
    )

epoch 01/20 train_loss=1.3384 test_loss=0.4156 test_top1=92.42% test_top5=97.36%
epoch 02/20 train_loss=0.3922 test_loss=0.4283 test_top1=93.10% test_top5=97.38%
epoch 03/20 train_loss=0.3603 test_loss=0.4121 test_top1=93.10% test_top5=97.30%
epoch 04/20 train_loss=0.3375 test_loss=0.4113 test_top1=92.34% test_top5=97.38%
epoch 05/20 train_loss=0.3655 test_loss=0.4343 test_top1=92.48% test_top5=97.74%
epoch 06/20 train_loss=0.4239 test_loss=0.4598 test_top1=92.59% test_top5=98.06%
epoch 07/20 train_loss=0.3871 test_loss=0.4163 test_top1=92.91% test_top5=97.66%
epoch 08/20 train_loss=0.3542 test_loss=0.4028 test_top1=93.13% test_top5=97.52%
epoch 09/20 train_loss=0.3463 test_loss=0.4170 test_top1=93.02% test_top5=97.36%
epoch 10/20 train_loss=0.3444 test_loss=0.4218 test_top1=92.83% test_top5=97.52%
epoch 11/20 train_loss=0.3436 test_loss=0.4095 test_top1=93.00% test_top5=97.63%
epoch 12/20 train_loss=0.3364 test_loss=0.4298 test_top1=93.13% test_top5=97.60%
epoch 13/20 train_loss=0.332

In [12]:
results_dir = OUT_ROOT / "probe"
results_dir.mkdir(parents=True, exist_ok=True)

final_metrics = evaluate(test_loader)
print("final test metrics:", final_metrics)

checkpoint = {
    "dataset": DATASET,
    "embed_dim": int(train_features.shape[1]),
    "num_classes": int(num_classes),
    "state_dict": probe.state_dict(),
    "train_mean": train_mean,
    "train_std": train_std,
    "epochs": PROBE_EPOCHS,
    "batch_size": PROBE_BATCH_SIZE,
    "lr": PROBE_LR,
    "weight_decay": PROBE_WEIGHT_DECAY,
    "test_metrics": final_metrics,
}

torch.save(checkpoint, results_dir / "linear_probe.pt")

metrics_json = {
    "dataset": DATASET,
    "embed_dim": int(train_features.shape[1]),
    "num_classes": int(num_classes),
    "epochs": PROBE_EPOCHS,
    "batch_size": PROBE_BATCH_SIZE,
    "lr": PROBE_LR,
    "weight_decay": PROBE_WEIGHT_DECAY,
    "test_loss": final_metrics["loss"],
    "test_top1": final_metrics["top1"],
    "test_top5": final_metrics["top5"],
}

with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_json, f, indent=2)

print(f"saved: {results_dir / 'linear_probe.pt'}")
print(f"saved: {results_dir / 'metrics.json'}")

final test metrics: {'loss': 0.4492565822003485, 'top1': 0.9302262196783865, 'top5': 0.9716544017656666}
saved: /Users/joshlichty/Repo/school/CS_547_open_clip_howeda_beanc_lichtyj/final-project/features/oxford_pet/probe/linear_probe.pt
saved: /Users/joshlichty/Repo/school/CS_547_open_clip_howeda_beanc_lichtyj/final-project/features/oxford_pet/probe/metrics.json
